In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
!pip install "transformers<5"
!pip install -q tf_keras
!pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!pip install  dglgo -f https://data.dgl.ai/wheels-test/repo.html
!pip install dgllife
!pip install torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install --pre deepchem

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import json
import time
import warnings
from scipy.signal import savgol_filter
import random
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import deepchem as dc
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import statistics
import dgl
import dgllife

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
warnings.filterwarnings("ignore")

# **PREPARE DATA**

In [ ]:
df = pd.read_csv("Dataset.csv")

smiles_col = "Canonical Smiles"
target_cols = list(df.loc[:, "220":"700"].columns)

print("SMILES column:", smiles_col)
print("Number of targets:", len(target_cols))


def random_split_dataframe(df, test_size=0.2, random_state=42):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        shuffle=True
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

def get_X_y_from_df(df, smiles_col, target_cols):
    X_smiles = df[smiles_col].astype(str).values
    y = df[target_cols].values.astype(np.float32)
    return X_smiles, y

train_random_df, test_random_df = random_split_dataframe(
    df,
    test_size=0.2
)
val_random_df, test_random_df = random_split_dataframe(
    test_random_df,
    test_size=0.5
)

print("\nRandom split:")
print("Train_df:", train_random_df.shape)
print("Val_df :", val_random_df.shape)
print("Test_df :", test_random_df.shape)

X_train, y_train = get_X_y_from_df(train_random_df, smiles_col, target_cols)
X_val, y_val = get_X_y_from_df(val_random_df, smiles_col, target_cols)
X_test, y_test = get_X_y_from_df(test_random_df, smiles_col, target_cols)

print("-" * 60)
print("Random train X shape    :", X_train.shape)
print("Random train y shape    :", y_train.shape)
print("Random val X shape      :", X_val.shape)
print("Random val y shape      :", y_val.shape)
print("Random test X shape     :", X_test.shape)
print("Random test y shape     :", y_test.shape)

In [ ]:
mgc_featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
X_train_mgc = mgc_featurizer.featurize(X_train)
X_val_mgc   = mgc_featurizer.featurize(X_val)
X_test_mgc  = mgc_featurizer.featurize(X_test)

train_dataset_attentivefp = dc.data.NumpyDataset(X=X_train_mgc, y=y_train)
val_dataset_attentivefp = dc.data.NumpyDataset(X=X_val_mgc, y=y_val)
test_dataset_attentivefp  = dc.data.NumpyDataset(X=X_test_mgc,  y=y_test)

print("AttentiveFP train dataset shape:", train_dataset_attentivefp.X.shape)

# **TRAINING**

In [ ]:
EPOCHS = 100
n_tasks = 481
MODEL_DIR_ROOT = "/content/results"
os.makedirs(MODEL_DIR_ROOT, exist_ok=True)

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def get_valid_pairs(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = ~np.isnan(a) & ~np.isnan(b)
    return a[mask], b[mask]

def match_coefficient(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    den = np.linalg.norm(a_valid) * np.linalg.norm(b_valid)
    return np.dot(a_valid, b_valid) / den

def pearson_correlation(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    return np.corrcoef(a_valid, b_valid)[0, 1]

def distance(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    return np.linalg.norm(a_valid - b_valid)

def safe_r2(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    return r2_score(a_valid, b_valid)

def safe_mae(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    return mean_absolute_error(a_valid, b_valid)

def safe_rmse(a, b):
    a_valid, b_valid = get_valid_pairs(a, b)
    return np.sqrt(mean_squared_error(a_valid, b_valid))

def regression_metrics(y_true, y_pred):
    r2_list, mae_list, rmse_list = [], [], []
    match_list, pearson_list, distance_list = [], [], []

    for i in range(y_true.shape[0]):
        yt = y_true[i, :]
        yp = y_pred[i, :]

        r2_list.append(safe_r2(yt, yp))
        mae_list.append(safe_mae(yt, yp))
        rmse_list.append(safe_rmse(yt, yp))
        match_list.append(match_coefficient(yt, yp))
        pearson_list.append(pearson_correlation(yt, yp))
        distance_list.append(distance(yt, yp))

    return {
        "median_r2": float(statistics.median(r2_list)),
        "median_mae": float(statistics.median(mae_list)),
        "median_rmse": float(statistics.median(rmse_list)),
        "median_match": float(statistics.median(match_list)),
        "median_pearson": float(statistics.median(pearson_list)),
        "median_distance": float(statistics.median(distance_list)),

        "avg_r2": float(np.mean(r2_list)),
        "avg_mae": float(np.mean(mae_list)),
        "avg_rmse": float(np.mean(rmse_list)),
        "avg_match": float(np.mean(match_list)),
        "avg_pearson": float(np.mean(pearson_list)),
        "avg_distance": float(np.mean(distance_list)),
    }

def train_and_validate_model(model, train_dataset, val_dataset, epochs=100):
    print("\n" + "#" * 90)
    print(f"TRAINING MODEL")
    print("#" * 90)
    print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
    print(model)

    model_dir = os.path.join(MODEL_DIR_ROOT)
    best_ckpt_dir = os.path.join(model_dir, "best_checkpoint")
    last_ckpt_dir = os.path.join(model_dir, "last_checkpoint")
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(best_ckpt_dir, exist_ok=True)
    os.makedirs(last_ckpt_dir, exist_ok=True)

    history = []
    start_time = time.time()

    best_val_rmse = float("inf")
    best_row = None

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        loss = model.fit(train_dataset, nb_epoch=1, loss=custom_loss_fn)

        y_train_pred = np.squeeze(to_numpy(model.predict(train_dataset)))
        y_val_pred   = np.squeeze(to_numpy(model.predict(val_dataset)))

        if y_train_pred.ndim == 1:
            y_train_pred = y_train_pred.reshape(-1, 1)
        if y_val_pred.ndim == 1:
            y_val_pred = y_val_pred.reshape(-1, 1)

        train_metrics = regression_metrics(train_dataset.y, y_train_pred)
        val_metrics   = regression_metrics(val_dataset.y, y_val_pred)

        row = {
            "epoch": epoch,
            "train_loss": float(loss),
            "val_median_r2": val_metrics["median_r2"],
            "val_median_mae": val_metrics["median_mae"],
            "val_median_rmse": val_metrics["median_rmse"],
            "val_median_match": val_metrics["median_match"],
            "val_median_pearson": val_metrics["median_pearson"],
            "val_median_distance": val_metrics["median_distance"],
            "val_avg_r2": val_metrics["avg_r2"],
            "val_avg_mae": val_metrics["avg_mae"],
            "val_avg_rmse": val_metrics["avg_rmse"],
            "val_avg_match": val_metrics["avg_match"],
            "val_avg_pearson": val_metrics["avg_pearson"],
            "val_avg_distance": val_metrics["avg_distance"],
            "train_avg_r2": train_metrics["avg_r2"],
            "train_avg_mae": train_metrics["avg_mae"],
            "train_avg_rmse": train_metrics["avg_rmse"],
            "train_avg_match": train_metrics["avg_match"],
            "train_avg_pearson": train_metrics["avg_pearson"],
            "train_avg_distance": train_metrics["avg_distance"],
            "train_median_r2": train_metrics["median_r2"],
            "train_median_mae": train_metrics["median_mae"],
            "train_median_rmse": train_metrics["median_rmse"],
            "train_median_match": train_metrics["median_match"],
            "train_median_pearson": train_metrics["median_pearson"],
            "train_median_distance": train_metrics["median_distance"],
            "epoch_time_sec": time.time() - epoch_start,
        }
        history.append(row)

        model.save_checkpoint(model_dir=last_ckpt_dir, max_checkpoints_to_keep=1)

        if row["val_median_rmse"] < best_val_rmse:
            best_val_rmse = row["val_median_rmse"]
            best_row = row.copy()
            model.save_checkpoint(model_dir=best_ckpt_dir, max_checkpoints_to_keep=1)

        print(
            f"[Epoch {epoch:03d}/{epochs} | "
            f"loss={row['train_loss']:.6f} | "
            f"val_median_RMSE={row['val_median_rmse']:.4f} | "
            f"val_median_MAE={row['val_median_mae']:.4f} | "
            f"val_median_R2={row['val_median_r2']:.4f} | "
            f"time={row['epoch_time_sec']:.2f}s"
        )

    history_df = pd.DataFrame(history)

    print("\nBest validation result:")
    print(json.dumps(best_row, indent=2))
    print(f"Total training time: {time.time() - start_time:.2f} sec")

    history_df.to_csv(os.path.join(model_dir, "history.csv"), index=False)

    return history_df, best_row

In [ ]:
class custom_loss(dc.models.losses.Loss):
    def _create_pytorch_loss(self):
        def loss_fn(output, labels):
            mask = torch.isfinite(labels)
            output = torch.where(mask, output, 0.0)
            labels = torch.where(mask, labels, 0.0)

            mse = ((output - labels) ** 2).sum() / mask.sum().clamp(min=1)

            dot = (output * labels).sum(dim=1)
            onorm = output.pow(2).sum(dim=1).sqrt()
            lnorm = labels.pow(2).sum(dim=1).sqrt()
            cos = 1 - (dot / (onorm * lnorm + 1e-8))[mask.any(dim=1)].mean()

            return mse + 0.2 * cos

        return loss_fn

In [6]:
model = dc.models.AttentiveFPModel(
        n_tasks=n_tasks,
        mode="regression",
        num_layers=3,
        num_timesteps=2,
        graph_feat_size=512,
        dense_layer_size=2048,
        dropout=0.1,
        batch_size=32,
        learning_rate=1e-3,
    )

custom = custom_loss()
custom_loss_fn = dc.models.torch_models.torch_model._StandardLoss(model, custom)

In [ ]:
history_df, best_row = train_and_validate_model(
    model=model,
    train_dataset=train_dataset_attentivefp,
    val_dataset=val_dataset_attentivefp,
    epochs=EPOCHS
)

# **INFERENCE**

In [ ]:
model_dir = os.path.join(MODEL_DIR_ROOT)
ckpt_dir = os.path.join(model_dir, "best_checkpoint") #"last_checkpoint"
model.restore(model_dir=ckpt_dir)

def get_valid_pairs_and_x(x, y_true, y_pred):
    x = np.asarray(x)
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return x[mask], y_true[mask], y_pred[mask], mask

In [ ]:
# Predict
y_test_pred = np.squeeze(to_numpy(model.predict(test_dataset_attentivefp)))

In [ ]:
i = random.randint(0, y_test.shape[0] - 1)
y_true_s = np.asarray(y_test[i], dtype=float)
y_pred_s = np.asarray(y_test_pred[i], dtype=float)
x = np.arange(220, 701)

x_valid, y_true_valid, y_pred_valid, mask = get_valid_pairs_and_x(x, y_true_s, y_pred_s)

mae_f = mean_absolute_error(y_true_valid, y_pred_valid)
rmse_f = np.sqrt(mean_squared_error(y_true_valid, y_pred_valid))
r2_f = r2_score(y_true_valid, y_pred_valid)
match_f = match_coefficient(y_true_valid, y_pred_valid)
pearson_f = pearson_correlation(y_true_valid, y_pred_valid)
distance_f = distance(y_true_valid, y_pred_valid)

fig, ax = plt.subplots(figsize=(12, 8))

y_pred_valid_smooth = savgol_filter(y_pred_valid, window_length=20, polyorder=2)
ax.plot(x_valid, y_true_valid, label='True', color='blue', linewidth=2)
ax.plot(x_valid, y_pred_valid_smooth, label='Predicted', color='red', linewidth=2)

ax.set_xticks(np.arange(220, 701, 50))
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Absorption')
ID = test_random_df["Compound (ID)"].values
ax.set_title(
    f'Compound {ID[i]}\n'
    f'RMSE={rmse_f:.4f}, Pearson={pearson_f:.4f}\n'
    f'Match={match_f:.4f}, , Distance={distance_f:.4f}'
)
ax.legend()

mol = Chem.MolFromSmiles(X_test[i])
if mol is not None:
    mol_img = Draw.MolToImage(mol, size=(300, 300))
    imagebox = OffsetImage(mol_img, zoom=0.5)
    ab = AnnotationBbox(
        imagebox,
        (0.75, 0.65),
        xycoords='axes fraction',
        frameon=True,
        pad=0.3,
        box_alignment=(0.5, 0.5)
    )
    ax.add_artist(ab)

plt.show()

In [ ]:
test_smiles = "OC1=CC=C(C=C1)/C=C/C2=CC(O)=CC(O)=C2OC3[C@@H]([C@H]([C@@H]([C@@H](C)O3)O)O)O"
test_X = mgc_featurizer.featurize([test_smiles])
test_dataset = dc.data.NumpyDataset(X=test_X)
test_y = model.predict(test_dataset)

y = test_y[0].astype(float)
x = np.linspace(220, 700, len(y))

y_smooth = savgol_filter(y, window_length=20, polyorder=2)
y_norm = (y_smooth - y_smooth.min()) / (y_smooth.max() - y_smooth.min())

mol = Chem.MolFromSmiles(test_smiles)
if mol is None:
    raise ValueError("Invalid SMILES string")

mol_img = Draw.MolToImage(mol, size=(300, 300))

fig, ax = plt.subplots(figsize=(12, 8))

#ax.plot(x, y, alpha=0.35, label="Raw")
ax.plot(x, y_norm, linewidth=2) #label="Smoothed + normalized")

ax.set_xlabel("Wavelength")
ax.set_ylabel("Intensity")
ax.set_title("Predicted UV-VIS spectra for test_smiles")
ax.legend()

imagebox = OffsetImage(np.array(mol_img), zoom=0.5)
ab = AnnotationBbox(
    imagebox,
    (0.82, 0.72),
    xycoords="axes fraction",
    frameon=True
)
ax.add_artist(ab)

plt.tight_layout()
plt.show()